In [9]:

import pandas as pd
import numpy as np
from pathlib import Path

In [10]:
DATA_PATH = Path("C:\my Files\RiskPredANDMarketEvaluation\data\processed\market_data.csv")
OUTPUT_DIR = Path("experiments/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [11]:
HORIZON = 10
THRESHOLD = 0.05


df = pd.read_csv(DATA_PATH, parse_dates=["date"])


accepted_tickers = ["NVDA", "AMD", "INTC", "TSM"]
df = df[df["ticker"].isin(accepted_tickers)].copy()

df = df.sort_values(["ticker", "date"]).reset_index(drop=True)


assert not df.duplicated(["date", "ticker"]).any()
assert (df["adj_close"].dropna() > 0).all()

print("Rows:", len(df))
print("Stocks:", sorted(df["ticker"].unique()))
print(df.head())

Rows: 5016
Stocks: ['AMD', 'INTC', 'NVDA', 'TSM']
        date ticker        open        high         low       close  \
0 2021-10-15    AMD  112.510002  112.839996  111.110001  112.120003   
1 2021-10-18    AMD  111.900002  116.879997  111.370003  116.430000   
2 2021-10-19    AMD  116.160004  117.169998  114.470001  116.330002   
3 2021-10-20    AMD  116.290001  118.300003  115.419998  116.389999   
4 2021-10-21    AMD  116.500000  119.680000  116.260002  119.330002   

    adj_close    volume  dividends  stock_splits  
0  112.120003  34203000        0.0           0.0  
1  116.430000  47537800        0.0           0.0  
2  116.330002  39593900        0.0           0.0  
3  116.389999  34244800        0.0           0.0  
4  119.330002  34794900        0.0           0.0  


In [12]:

def engineer_features(group):
    group = group.sort_values("date").copy()

    price = group["adj_close"]
    daily_return = price.pct_change(fill_method=None)

    # 1. Recent price returns
    group["return_1d"] = daily_return
    group["return_5d"] = price.pct_change(5, fill_method=None)
    group["return_10d"] = price.pct_change(10, fill_method=None)

    # 2. Recent volatility: standard deviation of daily returns
    group["volatility_5d"] = daily_return.rolling(5).std()
    group["volatility_20d"] = daily_return.rolling(20).std()

    # 3. Drawdown from the highest adjusted close seen so far
    running_peak = price.cummax()
    group["drawdown"] = price / running_peak - 1

    # 4. Trading volume behaviour
    previous_volume = group["volume"].shift(1)
    group["volume_change_1d"] = (
        group["volume"] / previous_volume - 1
    ).where(previous_volume > 0)

    # Compare today's volume with the previous 20 sessions
    previous_20d_volume = (
        group["volume"].shift(1).rolling(20).mean()
    )
    group["volume_ratio_20d"] = (
        group["volume"] / previous_20d_volume
    ).where(previous_20d_volume > 0)

    # 5. Target: does adjusted closing price fall at least 5%
    # below today's adjusted close at any of the NEXT 10 closes?
    future_prices = pd.concat(
        [
            price.shift(-step).rename(f"day_{step}")
            for step in range(1, HORIZON + 1)
        ],
        axis=1
    )

    complete_window = future_prices.notna().all(axis=1)
    future_min_close = future_prices.min(axis=1)

    group["stress_event_5pct_10d"] = np.nan
    group.loc[complete_window, "stress_event_5pct_10d"] = (
        future_min_close.loc[complete_window]
        <= price.loc[complete_window] * (1 - THRESHOLD)
    ).astype(int)

    return group


df_features = (
    df.groupby("ticker", group_keys=False)
      .apply(engineer_features)
      .reset_index(drop=True)
)

print(df_features[
    [
        "date", "ticker", "adj_close",
        "return_1d", "return_5d",
        "volatility_20d", "drawdown",
        "volume_ratio_20d", "stress_event_5pct_10d"
    ]
].tail(15))


           date ticker   adj_close  return_1d  return_5d  volatility_20d  \
5001 2026-09-24    TSM  451.149994   0.010256   0.048552        0.018072   
5002 2026-09-25    TSM  450.609985  -0.001197   0.036671        0.017546   
5003 2026-09-28    TSM  452.880005   0.005038   0.017388        0.016460   
5004 2026-09-29    TSM  456.940002   0.008965   0.010929        0.016330   
5005 2026-09-30    TSM  456.190002  -0.001641   0.021542        0.016293   
5006 2026-10-01    TSM  459.200012   0.006598   0.017843        0.016292   
5007 2026-10-02    TSM  472.779999   0.029573   0.049200        0.017164   
5008 2026-10-05    TSM  485.799988   0.027539   0.072690        0.017098   
5009 2026-10-06    TSM  482.299988  -0.007205   0.055500        0.016868   
5010 2026-10-07    TSM  472.200012  -0.020941   0.035095        0.017613   
5011 2026-10-08    TSM  457.989990  -0.030093  -0.002635        0.018670   
5012 2026-10-09    TSM  453.309998  -0.010219  -0.041182        0.018806   
5013 2026-10

C:\Users\aadib\AppData\Local\Temp\ipykernel_17008\3353639403.py:58: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(engineer_features)


In [13]:

feature_columns = [
    "return_1d",
    "return_5d",
    "return_10d",
    "volatility_5d",
    "volatility_20d",
    "drawdown",
    "volume_change_1d",
    "volume_ratio_20d",
]

print("Missing values in features:")
print(df_features[feature_columns].isna().sum())

print("\nTarget balance by stock:")
target_summary = (
    df_features.dropna(subset=["stress_event_5pct_10d"])
    .groupby("ticker")["stress_event_5pct_10d"]
    .agg(["count", "sum", "mean"])
)

target_summary["event_rate_pct"] = target_summary["mean"] * 100
print(target_summary.round(2))


Missing values in features:
return_1d            4
return_5d           20
return_10d          40
volatility_5d       20
volatility_20d      80
drawdown             0
volume_change_1d     4
volume_ratio_20d    80
dtype: int64

Target balance by stock:
        count    sum  mean  event_rate_pct
ticker                                    
AMD      1244  576.0  0.46           46.30
INTC     1244  555.0  0.45           44.61
NVDA     1244  492.0  0.40           39.55
TSM      1244  369.0  0.30           29.66


In [ ]:

# Pick one stock and a date somewhere inside its history
sample_ticker = "AMD"

sample = (
    df_features[df_features["ticker"] == sample_ticker]
    .sort_values("date")
    .reset_index(drop=True)
)

cutoff_position = len(sample) // 2
cutoff_date = sample.loc[cutoff_position, "date"]

# Recalculate features using only data up to that date
past_only = sample.loc[sample["date"] <= cutoff_date].copy()

recalculated = engineer_features(past_only)

# Compare the features on the cutoff date
original_row = sample.loc[
    sample["date"] == cutoff_date, feature_columns
].iloc[0]

recalculated_row = recalculated.loc[
    recalculated["date"] == cutoff_date, feature_columns
].iloc[0]

comparison = pd.DataFrame({
    "original": original_row,
    "recalculated_using_past_only": recalculated_row,
})

comparison["absolute_difference"] = (
    comparison["original"] - comparison["recalculated_using_past_only"]
).abs()

print(comparison)
print(
    "\nAll features unchanged:",
    np.allclose(
        comparison["original"].to_numpy(dtype=float),
        comparison["recalculated_using_past_only"].to_numpy(dtype=float),
        equal_nan=True,
        atol=1e-10,
    )
)


In [19]:

# Independently reconstruct the target from adjusted closing prices
# and compare it with the saved target labels.

check = df_features.sort_values(
    ["ticker", "date"]
).reset_index(drop=True)

reconstructed_labels = pd.Series(
    np.nan, index=check.index, dtype=float
)

for _, indices in check.groupby("ticker", sort=False).groups.items():
    indices = list(indices)
    prices = check.loc[indices, "adj_close"].reset_index(drop=True)

    future_prices = pd.concat(
        [prices.shift(-step) for step in range(1, 11)],
        axis=1
    )

    complete = future_prices.notna().all(axis=1)
    future_min = future_prices.min(axis=1)

    labels = pd.Series(np.nan, index=prices.index, dtype=float)
    labels.loc[complete] = (
        future_min.loc[complete]
        <= prices.loc[complete] * 0.95
    ).astype(int)

    reconstructed_labels.loc[indices] = labels.to_numpy()

target_column="stress_event_5pct_10d"
actual = check[target_column].reset_index(drop=True)
expected = reconstructed_labels.reset_index(drop=True)

comparable = actual.notna() & expected.notna()

print("Comparable target rows:", comparable.sum())
print(
    "Target mismatches:",
    (actual[comparable] != expected[comparable]).sum()
)
print(
    "Target missingness matches:",
    actual.isna().equals(expected.isna())
)


Comparable target rows: 4976
Target mismatches: 0
Target missingness matches: True


In [21]:

model_columns = [
    "date",
    "ticker",
    *feature_columns,
    target_column,
]

# Keep rows that have all features and a known target
model_data = df_features[model_columns].copy()

# Convert any infinities to missing values before filtering
model_data[feature_columns] = (
    model_data[feature_columns]
    .replace([np.inf, -np.inf], np.nan)
)

before = len(model_data)

model_data = model_data.dropna(
    subset=feature_columns + [target_column]
).copy()

model_data[target_column] = (
    model_data[target_column].astype("int8")
)

print("Rows before filtering:", before)
print("Rows available for modeling:", len(model_data))
print("\nRows per ticker:")
print(model_data.groupby("ticker").size())

output_path = OUTPUT_DIR / "model_ready_candidates.csv"
model_data.to_csv(output_path, index=False)

print("\nSaved to:", output_path)


Rows before filtering: 5016
Rows available for modeling: 4896

Rows per ticker:
ticker
AMD     1224
INTC    1224
NVDA    1224
TSM     1224
dtype: int64

Saved to: C:\my Files\risk pred & market evaluation\experiments\outputs\model_ready_candidates.csv
